# Overnight-range breakout trade selection

This research notebook creates a causal session-level feature dataframe, then tests whether LightGBM can select breakout trades with higher risk-normalized return than the baseline of taking every eligible trade. The newest 20% of rows remain untouched until final scoring.

In [38]:
from datetime import timedelta, datetime
from zoneinfo import ZoneInfo

import numpy as np
import polars as pl
from lightgbm import LGBMRegressor

LONDON = ZoneInfo("Europe/London")

from libs.overnight_range import (
    OvernightRangeBreakoutConfig,
    OvernightRangeFeatureConfig,
    build_overnight_range_breakout_ledger,
    build_overnight_range_features,
)

DATA_PATH = r"C:/Code/Trading/2026/ml/data/minute_bars_gbpusd.parquet"
FEATURE_EXPORT_PATH = None  # e.g. r"C:/Code/Trading/2026/ml/data/gbpusd_overnight_breakout_features.parquet"

BREAKOUT_CONFIG = OvernightRangeBreakoutConfig(
    buffer_pips=3,
    stop_range_multiple=0.5,
    target_range_multiple=2.0,
)
FEATURE_CONFIG = OvernightRangeFeatureConfig(
    daily_range_windows=(5, 20),
    moving_average_window=20,
    minimum_range_coverage=1.0,
    minimum_trade_coverage=1.0,
)

TEST_FRACTION = 0.20
VALIDATION_FRACTION_OF_DEVELOPMENT = 0.10
MINIMUM_VALIDATION_COVERAGE = 0.10


In [39]:
all_bars = pl.read_parquet(DATA_PATH)
bars = all_bars.filter((pl.datetime(2005, 1, 1, time_zone='UTC') <=pl.col("timestamp")) & (pl.col("timestamp") < pl.datetime(2020, 1, 1, time_zone='UTC')))
ledger = build_overnight_range_breakout_ledger(bars, BREAKOUT_CONFIG)
feature_df, feature_names = build_overnight_range_features(ledger, bars, FEATURE_CONFIG)

if FEATURE_EXPORT_PATH:
    feature_df.write_parquet(FEATURE_EXPORT_PATH, compression="zstd")

print(f"Feature rows: {feature_df.height:,}; model features: {len(feature_names)}")
feature_df.select(["session_date", "side", "target", "gross_pnl_pips", *feature_names]).head()

Feature rows: 1,351; model features: 23


session_date,side,target,gross_pnl_pips,range_pips,overnight_return_to_range,overnight_close_position,previous_daily_range_pips,previous_daily_return_to_range,range_to_previous_daily_range,range_to_previous_overnight_range,overnight_high_position_in_previous_day,overnight_low_position_in_previous_day,overnight_inside_previous_day,side_is_long,entry_with_previous_day_direction,entry_hour_london,entry_minute_london,entry_to_ma_pips,previous_close_to_ma_pips,range_to_adr_5,range_to_adr_20,weekday_1,weekday_2,weekday_3,weekday_4,weekday_5
date,str,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,f64,f64,f64,f64,i8,i8,i8,i8,i8
2005-01-25,"""short""",0,-25.775,51.55,-0.520854,0.078565,96.1,0.184183,0.53642,1.164972,0.705515,0.169095,1,0,0,7,5,-30.6,2.6,0.413425,0.359346,0,1,0,0,0
2005-01-26,"""long""",1,113.7,65.1,0.602151,0.608295,182.8,-0.849836,0.356127,1.262852,0.413567,0.05744,1,1,0,9,22,-50.285,-124.885,0.533563,0.432214,0,0,1,0,0
2005-01-28,"""short""",0,-31.375,62.75,-0.521116,0.443825,126.8,0.778391,0.494874,1.588608,0.843849,0.348975,1,0,0,11,21,84.745,155.345,0.482062,0.441489,0,0,0,0,1
2005-02-01,"""short""",1,50.7,25.35,0.42998,0.473373,94.3,-0.18982,0.268823,0.700276,0.800106,0.531283,1,0,1,7,13,69.795,76.995,0.214558,0.19898,0,1,0,0,0
2005-02-02,"""long""",0,-17.075,34.15,0.38653,0.45388,86.95,-0.111558,0.392754,1.34714,0.989649,0.596895,1,1,0,11,7,107.695,62.195,0.371721,0.261771,0,0,1,0,0


In [40]:
# Chronological development / final-test split. Threshold selection uses only
# the final slice of development data, never the final test period.
test_start = int(feature_df.height * (1 - TEST_FRACTION))
development, test = feature_df[:test_start], feature_df[test_start:]
validation_start = int(development.height * (1 - VALIDATION_FRACTION_OF_DEVELOPMENT))
fit, validation = development[:validation_start], development[validation_start:]

model_parameters = dict(
    objective="regression", n_estimators=300, learning_rate=0.03, num_leaves=31,
    min_child_samples=30, reg_lambda=1.0, random_state=18616, verbosity=-1,
)
validation_model = LGBMRegressor(**model_parameters).fit(
    fit.select(feature_names).to_numpy(), fit["r_return"].to_numpy()
)
validation_prediction = validation_model.predict(validation.select(feature_names).to_numpy())

candidates = []
for threshold in np.quantile(validation_prediction, np.linspace(0.00, 0.90, 46)):
    selected = validation.filter(pl.Series(validation_prediction >= threshold))
    coverage = selected.height / validation.height
    if coverage >= MINIMUM_VALIDATION_COVERAGE:
        candidates.append((selected["r_return"].mean(), coverage, threshold))

if not candidates:
    raise ValueError("No validation threshold meets MINIMUM_VALIDATION_COVERAGE.")
selected_threshold = max(candidates)[2]
print(f"Selected validation predicted-R threshold: {selected_threshold:.3f}")

# Refit using all development rows, then score the untouched newest period.
final_model = LGBMRegressor(**model_parameters).fit(
    development.select(feature_names).to_numpy(), development["r_return"].to_numpy()
)
test_prediction = final_model.predict(test.select(feature_names).to_numpy())
selected_test = test.filter(pl.Series(test_prediction >= selected_threshold))

def performance(name, frame):
    return {
        "strategy": name,
        "trades": frame.height,
        "coverage": frame.height / test.height,
        "win_rate": frame["target"].mean(),
        "total_r_return": frame["r_return"].sum(),
        "mean_r_return": frame["r_return"].mean(),
    }

comparison = pl.DataFrame([performance("baseline_all_trades", test), performance("lightgbm_selected", selected_test)])
comparison

Selected validation predicted-R threshold: -0.184


strategy,trades,coverage,win_rate,total_r_return,mean_r_return
str,i64,f64,f64,f64,f64
"""baseline_all_trades""",271,1.0,0.261993,-13.578476,-0.050105
"""lightgbm_selected""",163,0.601476,0.294479,11.892028,0.072957


In [41]:
# NEW_DATA must be strictly after the data used to fit final_model. Keep prior
# bars solely for the causal 20-day daily-history features.
new_bars = all_bars.filter((pl.datetime(2020, 1, 1, time_zone='UTC') <= pl.col("timestamp")) & (pl.col("timestamp") < pl.datetime(2021, 1, 1, time_zone='UTC')))
new_start = new_bars["timestamp"].min()
new_end = new_bars["timestamp"].max()
context_bars = all_bars.filter(pl.col("timestamp").is_between(new_start - timedelta(days=35), new_end, closed="both"))
scoring_bars = context_bars.sort("timestamp")
scoring_ledger = build_overnight_range_breakout_ledger(scoring_bars, BREAKOUT_CONFIG)
scoring_feature_df, scoring_feature_names = build_overnight_range_features(
    scoring_ledger, scoring_bars, FEATURE_CONFIG
)
new_start_date = new_start.astimezone(LONDON).date()
scoring_feature_df = scoring_feature_df.filter(pl.col("session_date") >= new_start_date)
if scoring_feature_names != feature_names:
    raise ValueError("Scoring feature schema differs from the fitted model.")
print(f"Scorable new rows: {scoring_feature_df.height:,}")

Scorable new rows: 40


In [42]:
if scoring_feature_df.is_empty():
    raise ValueError("No scorable new rows. Check that new data contains entered trades and enough prior context.")

test_prediction = final_model.predict(scoring_feature_df.select(feature_names).to_numpy())
scored_new_data = scoring_feature_df.with_columns([
    pl.Series("predicted_r_return", test_prediction),
    (pl.Series(test_prediction) >= selected_threshold).alias("take_trade"),
])
scored_new_data.select(["session_date", "side", "predicted_r_return", "take_trade", "r_return"])


session_date,side,predicted_r_return,take_trade,r_return
date,str,f64,bool,f64
2020-08-26,"""long""",-0.00594,true,-1.0
2020-08-27,"""long""",0.677258,true,-1.0
2020-08-28,"""long""",-0.651418,false,1.430707
2020-09-01,"""long""",-0.193997,false,-0.295345
2020-09-02,"""short""",-1.159527,false,4.0
…,…,…,…,…
2020-12-10,"""short""",0.962762,true,1.991919
2020-12-18,"""short""",-0.299055,false,-1.0
2020-12-23,"""long""",0.41098,true,4.0


In [43]:
scored_new_data['r_return'].sum()

13.099572243244038